In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset
import time
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split


import torch
import pyro
import pyro.distributions as dist
from pyro.nn.module import PyroModule, PyroParam
from pyro.infer.autoguide import AutoGuide
from pyro.infer.autoguide.initialization import InitMessenger, init_to_feasible
from pyro.distributions import constraints
from contextlib import ExitStack

import pickle
from tqdm import tqdm
import copy

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
from pyro.infer.autoguide import AutoNormal

import pandas as pd

import numpy as np
from sklearn.metrics import confusion_matrix

from bitflip import bitflip_float32

from torchvision.datasets import ImageFolder

import os

import json

shipsnet_mean = [0.4119, 0.4243, 0.3724]
shipsnet_std = [0.1899, 0.1569, 0.1515]

def load_data(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize(mean=shipsnet_mean, 
                             std=shipsnet_std)
    ])

    #dataset = datasets.EuroSAT(root='./data', transform=transform, download=True)
    dataset = ImageFolder(
    root="data/shipsnet/foldered",
    transform=transform
    )
    torch.manual_seed(42)

    #train_size = int(0.8 * len(dataset))
    #test_size = len(dataset) - train_size
    #train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    with open('datasplit/shipsnet_split_indices.pkl', 'rb') as f:
        split = pickle.load(f)
        train_dataset = Subset(dataset, split['train'])
        test_dataset = Subset(dataset, split['test'])

    # Add num_workers and pin_memory for faster data loading
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            num_workers=4, pin_memory=True, persistent_workers=True)
    return train_loader, test_loader

from pyro.distributions.util       import sum_rightmost
from pyro.ops.tensor_utils         import periodic_repeat
from pyro.distributions.transforms import biject_to
from pyro.infer.autoguide.utils    import (
    deep_setattr,
    deep_getattr,
    helpful_support_errors,
)

import pyro.poutine as poutine


class AutoLaplace(AutoGuide):
    """
    An AutoGuide that uses a Laplace(loc, scale) marginal for each latent.
    """
    scale_constraint = constraints.softplus_positive

    def __init__(
        self, model, *, init_loc_fn=init_to_feasible, init_scale=0.1, create_plates=None
    ):
        self.init_loc_fn = init_loc_fn
        if not isinstance(init_scale, float) or not (init_scale > 0):
            raise ValueError(f"Expected init_scale > 0, got {init_scale}")
        self._init_scale = init_scale

        model = InitMessenger(self.init_loc_fn)(model)
        super().__init__(model, create_plates=create_plates)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)

        self._event_dims = {}
        self.locs = PyroModule()
        self.scales = PyroModule()

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            # ← use helpful_support_errors directly
            with helpful_support_errors(site):
                init_loc = (
                    biject_to(site["fn"].support)
                    .inv(site["value"].detach())
                    .detach()
                )
            event_dim = site["fn"].event_dim + init_loc.dim() - site["value"].dim()
            self._event_dims[name] = event_dim

            # handle subsampling plates
            for frame in site["cond_indep_stack"]:
                full_size = frame.full_size or frame.size
                if full_size != frame.size:
                    dim = frame.dim - event_dim
                    init_loc = periodic_repeat(init_loc, full_size, dim).contiguous()

            init_scale = torch.full_like(init_loc, self._init_scale)

            deep_setattr(
                self.locs, name, PyroParam(init_loc, constraints.real, event_dim)
            )
            deep_setattr(
                self.scales,
                name,
                PyroParam(init_scale, self.scale_constraint, event_dim),
            )

    def _get_loc_and_scale(self, name):
        site_loc = deep_getattr(self.locs, name)
        site_scale = deep_getattr(self.scales, name)
        return site_loc, site_scale

    def forward(self, *args, **kwargs):
        if self.prototype_trace is None:
            self._setup_prototype(*args, **kwargs)

        plates = self._create_plates(*args, **kwargs)
        result = {}

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            transform = biject_to(site["fn"].support)

            with ExitStack() as stack:
                for frame in site["cond_indep_stack"]:
                    if frame.vectorized:
                        stack.enter_context(plates[frame.name])

                site_loc, site_scale = self._get_loc_and_scale(name)

                unconstrained = pyro.sample(
                    f"{name}_unconstrained",
                    dist.Laplace(site_loc, site_scale)
                        .to_event(self._event_dims[name]),
                    infer={"is_auxiliary": True},
                )

                value = transform(unconstrained)
                if poutine.get_mask() is False:
                    log_density = 0.0
                else:
                    log_density = transform.inv.log_abs_det_jacobian(
                        value, unconstrained
                    )
                    log_density = sum_rightmost(
                        log_density,
                        log_density.dim() - value.dim() + site["fn"].event_dim,
                    )
                delta = dist.Delta(
                    value,
                    log_density=log_density,
                    event_dim=site["fn"].event_dim,
                )
                result[name] = pyro.sample(name, delta)

        return result

    @torch.no_grad()
    def median(self, *args, **kwargs):
        medians = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            site_loc, _ = self._get_loc_and_scale(name)
            med = biject_to(site["fn"].support)(site_loc)
            medians[name] = med.clone() if med is site_loc else med
        return medians

    @torch.no_grad()
    def quantiles(self, quantiles, *args, **kwargs):
        results = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            site_loc, site_scale = self._get_loc_and_scale(name)
            qs = torch.tensor(quantiles, dtype=site_loc.dtype, device=site_loc.device)
            qs = qs.reshape((-1,) + (1,) * site_loc.dim())
            qvals = dist.Laplace(site_loc, site_scale).icdf(qs)
            results[name] = biject_to(site["fn"].support)(qvals)
        return results


class AutoUniform(AutoGuide):
    """
    An AutoGuide that uses a Uniform(low, low+width) marginal for each latent.
    """
    # `width` must be positive
    width_constraint = constraints.softplus_positive

    def __init__(
        self, model, *, init_loc_fn=init_to_feasible, init_scale=0.1, create_plates=None
    ):
        self.init_loc_fn = init_loc_fn
        if not isinstance(init_scale, float) or not (init_scale > 0):
            raise ValueError(f"Expected init_scale > 0, got {init_scale}")
        self._init_scale = init_scale

        model = InitMessenger(self.init_loc_fn)(model)
        super().__init__(model, create_plates=create_plates)

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        self._event_dims = {}
        self.lows = PyroModule()
        self.widths = PyroModule()

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            # 1. get an unconstrained init_loc (inverse‐transform of site["value"])
            with helpful_support_errors(site):
                init_loc = (
                    biject_to(site["fn"].support)
                    .inv(site["value"].detach())
                    .detach()
                )
            event_dim = site["fn"].event_dim + init_loc.dim() - site["value"].dim()
            self._event_dims[name] = event_dim

            # 2. if subsampled, expand back to full size
            for frame in site["cond_indep_stack"]:
                full_size = frame.full_size or frame.size
                if full_size != frame.size:
                    dim = frame.dim - event_dim
                    init_loc = periodic_repeat(init_loc, full_size, dim).contiguous()

            # 3. build initial low & width around that init_loc
            init_low   = init_loc - self._init_scale
            init_width = torch.full_like(init_loc, 2.0 * self._init_scale)

            # 4. register as PyroParams
            deep_setattr(
                self.lows,  name,
                PyroParam(init_low,   constraints.real,             event_dim),
            )
            deep_setattr(
                self.widths, name,
                PyroParam(init_width, self.width_constraint,       event_dim),
            )

    def _get_low_and_width(self, name):
        low   = deep_getattr(self.lows,  name)
        width = deep_getattr(self.widths, name)
        return low, width

    def forward(self, *args, **kwargs):
        if self.prototype_trace is None:
            self._setup_prototype(*args, **kwargs)

        plates = self._create_plates(*args, **kwargs)
        result = {}

        for name, site in self.prototype_trace.iter_stochastic_nodes():
            transform = biject_to(site["fn"].support)
            with ExitStack() as stack:
                for frame in site["cond_indep_stack"]:
                    if frame.vectorized:
                        stack.enter_context(plates[frame.name])

                low, width = self._get_low_and_width(name)
                # draw unconstrained latent from Uniform(low, low + width)
                unconstrained = pyro.sample(
                    f"{name}_unconstrained",
                    dist.Uniform(low, low + width).to_event(self._event_dims[name]),
                    infer={"is_auxiliary": True},
                )

                # map into constrained space
                value = transform(unconstrained)
                if poutine.get_mask() is False:
                    log_density = 0.0
                else:
                    log_density = transform.inv.log_abs_det_jacobian(
                        value, unconstrained
                    )
                    log_density = sum_rightmost(
                        log_density,
                        log_density.dim() - value.dim() + site["fn"].event_dim,
                    )
                delta = dist.Delta(
                    value,
                    log_density=log_density,
                    event_dim=site["fn"].event_dim,
                )
                result[name] = pyro.sample(name, delta)

        return result

    @torch.no_grad()
    def median(self, *args, **kwargs):
        """
        Posterior median is just the 0.5‐quantile of Uniform = low + 0.5*width
        """
        medians = {}
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            low, width = self._get_low_and_width(name)
            med = biject_to(site["fn"].support)(low + 0.5 * width)
            medians[name] = med.clone() if med is low else med
        return medians

    @torch.no_grad()
    def quantiles(self, quantiles, *args, **kwargs):
        """
        Posterior quantiles via Uniform.icdf(q).
        """
        results = {}
        qs = torch.tensor(quantiles)
        for name, site in self.prototype_trace.iter_stochastic_nodes():
            low, width = self._get_low_and_width(name)
            # shape: [len(quantiles), *low.shape]
            qvals = dist.Uniform(low, low + width).icdf(qs.reshape((-1,) + (1,) * low.dim()))
            results[name] = biject_to(site["fn"].support)(qvals)
        return results


class BayesShipsCNN(PyroModule):
    def __init__(
        self,
        num_classes=2,   # now 2 for Categorical
        device=torch.device("cuda"),
        activation='relu',
        prior_dist='gaussian',
        mu=0.0,
        b=1.0,
        prior_params=None
    ):
        super().__init__()
        device = device

        # Activation setup
        if isinstance(activation, str):
            act_map = {
                'relu': F.relu,
                'tanh': torch.tanh,
                'sigmoid': torch.sigmoid,
                'sin': torch.sin,
                'relu6': F.relu6,
                'leaky_relu': F.leaky_relu,
                'selu': F.selu,
                'actWG': self.actWG,
                'actRWG': self.actRWG,
            }
            self.activation_fn = act_map[activation]
        elif callable(activation):
            self.activation_fn = activation
        else:
            raise ValueError("activation must be a string or callable")

        # Prior setup
        self.prior_dist = prior_dist
        params = {'mu': mu, 'b': b} if prior_params is None else prior_params
        self.prior_mu = torch.tensor(params['mu'], device=device, dtype=torch.float32)
        self.prior_b  = torch.tensor(params['b'], device=device, dtype=torch.float32)

        print(f"[INFO] Using prior: {self.prior_dist} (mu={self.prior_mu.item()}, b={self.prior_b.item()})")

        # Layers
        self.conv1 = PyroModule[nn.Conv2d](3, 32, kernel_size=3, padding=1)
        self.conv1.weight = PyroSample(self._make_prior([32, 3, 3, 3]))
        self.conv1.bias   = PyroSample(self._make_prior([32]))

        self.conv2 = PyroModule[nn.Conv2d](32, 64, kernel_size=3, padding=1)
        self.conv2.weight = PyroSample(self._make_prior([64, 32, 3, 3]))
        self.conv2.bias   = PyroSample(self._make_prior([64]))

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = PyroModule[nn.Linear](64 * 16 * 16, num_classes)
        self.fc1.weight = PyroSample(self._make_prior([num_classes, 64 * 16 * 16]))
        self.fc1.bias   = PyroSample(self._make_prior([num_classes]))

    def actWG(self, x, alpha=1.0):
        return x * torch.exp(-alpha * x ** 2)

    def actRWG(self, x, alpha=1.0):
        wg = x * torch.exp(-alpha * x ** 2)
        return torch.max(torch.zeros_like(wg), wg)

    def _make_prior(self, shape):
        if self.prior_dist == 'gaussian':
            base = dist.Normal(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'laplace':
            base = dist.Laplace(self.prior_mu, self.prior_b)
        elif self.prior_dist == 'uniform':
            base = dist.Uniform(-self.prior_b, self.prior_b)
        else:
            raise ValueError(f"Unsupported prior: {self.prior_dist}")
        return base.expand(shape).to_event(len(shape))

    def forward(self, x, y=None):
        x = self.activation_fn(self.conv1(x))
        x = self.pool(x)
        x = self.activation_fn(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        logits = self.fc1(x)  # shape [batch, 2]

        if y is not None:
            with pyro.plate("data", x.size(0)):
                pyro.sample("obs", dist.Categorical(logits=logits), obs=y)
        return logits


def load_model(timestamp):
    config_path = os.path.join(search_dir, config_files[timestamp])
    guide_path = os.path.join(search_dir, guide_files[timestamp])
    model_path = os.path.join(search_dir, model_files[timestamp])
    param_path = os.path.join(search_dir, param_files[timestamp])

    print(f"Loading model with config_path: {config_path}")

    with open(config_path, 'r') as f:
        config = json.load(f)

    model = BayesShipsCNN(
        num_classes=num_classes,
        device=device,
        activation=config['activation'],
        prior_dist=config['prior'],
        mu=config['prior_params']['mu'],
        b=config['prior_params'],
        prior_params=config.get('prior_params', None)
    ).to(device)

    # Load the guide
    #guide = AutoDiagonalNormal(model)

    # Load the model state
    model.load_state_dict(torch.load(model_path))
    
    # Load the guide state
    #guide.load_state_dict(torch.load(guide_path))

    return model, param_path

c:\Users\Revalda Putawara\.conda\envs\bnntest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_loader, test_loader = load_data(batch_size=16)
device = torch.device("cuda")
num_classes = 2

In [3]:
search_dir = "results_GP_shipsnet_newslate_guide"

def load_model_config(timestamp):
    config_path = os.path.join(search_dir, config_files[timestamp])

    with open(config_path, 'r') as f:
        model_config = json.load(f)

    return model_config

In [4]:
save_dir = "shipsnet_seu_result"

search_dir = "results_GP_shipsnet_newslate_guide"
#list all .json files in the directory
all_files = [f for f in os.listdir(search_dir)]
json_files = [f for f in os.listdir(search_dir) if f.endswith('.json')]

# excluding the format, get the last 16 characters of each filename
timestamps = [f[:-5][-16:] for f in json_files]
print("Timestamps found count:", len(timestamps))

# remove some timestamps that are not needed
# those are the ones that are not in the shipsnet_seu_result directory, without the .csv extension
excluded_timestamps = [f[:-4][-16:] for f in [f for f in os.listdir(save_dir) if f.endswith('.csv')]]

timestamps = [ts for ts in timestamps if ts not in excluded_timestamps]
#timestamps = timestamps[:1]
print("After excluding, timestamps count:", len(timestamps))

# for each timestamp, look for every other files in the directory that contains the timestamp

config_files = {}
guide_files = {}
model_files = {}
param_files = {}

for timestamp in timestamps:
    config_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('.json')][0]
    guide_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('guide')][0]
    model_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('model')][0]
    param_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('param')][0]

# create a list that maps timestamps and the output of load_model_config['prior']
prior_list = []
for ts in timestamps:
    model_config = load_model_config(ts)
    prior_list.append((ts, model_config['prior']))

Timestamps found count: 63
After excluding, timestamps count: 63


In [5]:


prior_map = {
    'Gaussian_prior': 'gaussian',
    'Laplace_prior': 'laplace',
    'Uniform_prior': 'uniform'
}
#timestamps = [ts for ts, prior in prior_list if prior == prior_map[args.prior]]
#print(f"After filtering by prior '{args.prior}', timestamps count: {len(timestamps)}")

In [38]:
timestamps_filtered = timestamps[1]  # For testing, limit to one timestamp
timestamps_filtered = '_20250718_234102'

In [39]:
pyro.clear_param_store()

bayesian_model, pyro_param_store_path = load_model(timestamps_filtered)

Loading model with config_path: results_GP_shipsnet_newslate_guide\config_relu_gaussian_20250718_234102.json
[INFO] Using prior: gaussian (mu=0.0, b=1.0)


In [40]:
if bayesian_model.prior_dist == 'gaussian':
    guide = AutoNormal(bayesian_model, init_scale=0.05).to(device)
elif bayesian_model.prior_dist == 'laplace':
    guide = AutoLaplace(bayesian_model, init_scale=0.05).to(device)
elif bayesian_model.prior_dist == 'uniform':
    guide = AutoUniform(bayesian_model, init_scale=0.05).to(device)
else:
    raise ValueError(f"Unsupported prior: {bayesian_model.prior_dist}")

In [41]:
bayesian_model.eval()

BayesShipsCNN(
  (conv1): PyroConv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): PyroConv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): PyroLinear(in_features=16384, out_features=2, bias=True)
)

In [42]:
pyro.get_param_store().clear()
pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

In [43]:
def predict_data_probs(self, num_samples=10):
    all_labels = []
    all_predictions = []
    all_logits = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            logits_mc = torch.zeros(num_samples, images.size(0), bayesian_model.fc1.out_features).to(device)

            for i in range(num_samples):
                guide_trace = pyro.poutine.trace(guide).get_trace(images)
                replayed_model = pyro.poutine.replay(bayesian_model, trace=guide_trace)
                logits = replayed_model(images)
                logits_mc[i] = logits

            avg_logits = logits_mc.mean(dim=0)
            predictions = torch.argmax(avg_logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_logits.extend(avg_logits.cpu().numpy())
            all_probs.extend(F.softmax(avg_logits, dim=1).cpu().numpy())

    return all_labels, all_predictions, all_logits, all_probs

In [44]:
def return_accuracy(all_labels, all_predictions):
    cm = confusion_matrix(all_labels, all_predictions)
    return np.trace(cm) / np.sum(cm)

In [45]:
num_samples = 10

initial_labels, initial_predictions, initial_logits, initial_probs = predict_data_probs(num_samples)
initial_accuracy = return_accuracy(initial_labels, initial_predictions)
initial_probs = np.array(initial_probs)

Evaluating: 100%|██████████| 50/50 [00:04<00:00, 12.28it/s]


In [55]:
import numpy as np
import torch
import torch.nn.functional as F

def compute_softmax_difference(before_probs, after_logits, penalty=1.0):
    """
    before_probs: list or array, shape (N, C), all finite probabilities
    after_logits: list or array, shape (N, C), raw logits (may contain ±inf)
    penalty: float, the per‑example penalty to use if logits are nonfinite
    
    Returns the mean over N examples of either
      - max_i |before_probs[n,i] − after_probs[n,i]|,  if after_logits[n] is finite
      - penalty,                                    otherwise
    """
    before = np.asarray(before_probs, dtype=np.float32)
    after_logits = torch.from_numpy(np.asarray(after_logits, dtype=np.float32))
    N, C = after_logits.shape

    # 1) detect which rows of after_logits are all finite
    finite_mask = torch.isfinite(after_logits).all(dim=1).numpy()  # shape (N,)

    # 2) safe‑softmax only on the finite ones
    safe_after_probs = torch.zeros_like(after_logits)
    if finite_mask.any():
        good_logits = after_logits[finite_mask]
        # (you can optionally do the “stable” shift here)
        safe_after_probs[finite_mask] = F.softmax(good_logits, dim=1)
    safe_after_probs = safe_after_probs.numpy()

    # 3) compute per‑example diff, using the penalty where needed
    diffs = np.empty(N, dtype=np.float32)
    for n in range(N):
        if not finite_mask[n]:
            diffs[n] = penalty
        else:
            diffs[n] = np.max(np.abs(before[n] - safe_after_probs[n]))
    return diffs.mean()


def compute_difference(original_val, modified_val):
    return abs(original_val - modified_val)

In [56]:
initial_accuracy

np.float64(0.89875)

In [57]:
def run_seu(location_index, param_unique, parameter_name, layer, layer_module, bit_i, num_samples):
    assert parameter_name in ["locs", "scales", "lows", "widths"], "Parameter name must be 'locs' or 'scales'."
    assert bit_i in range(0, 33), "Bit index must be between 0 and 32."

    param_store_name = f"{param_unique}.{parameter_name}.{layer}.{layer_module}"
    pyro.get_param_store().set_state(torch.load(pyro_param_store_path, weights_only=False))

    with torch.no_grad():
        param = pyro.get_param_store().get_param(param_store_name)
        new_param = param.clone()
        new_param = new_param.view(-1) #flatten new param
        original_val = new_param[location_index].cpu().item()
        seu_val = bitflip_float32(original_val, bit_i)
        abs_diff = compute_difference(original_val, seu_val)
        new_param[location_index] = seu_val
        # return new_param to original shape
        new_param = new_param.view(param.shape)
        pyro.get_param_store().__setitem__(param_store_name, new_param)

        print(f"Original value: {original_val}, SEU value: {seu_val}, Abs difference: {abs_diff}")


    if param_unique == "AutoNormal":
        guide = AutoNormal(bayesian_model, init_scale=0.05).to(device)
    elif param_unique == "AutoLaplace":
        guide = AutoLaplace(bayesian_model, init_scale=0.05).to(device)
    elif param_unique == "AutoUniform":
        guide = AutoUniform(bayesian_model, init_scale=0.05).to(device)
    else:
        raise ValueError(f"Unsupported parameter unique: {param_unique}")

    try:
        after_labels, after_predictions, after_logits, after_probs = predict_data_probs(num_samples)
        accuracy_after = return_accuracy(after_labels, after_predictions)
        softmax_diff = compute_softmax_difference(initial_probs, after_probs)

        print(f"Initial logits: {initial_logits}")
        print(f"Initial probs: {initial_probs}")
        print(f"After logits: {after_logits}")
        print(f"After probs: {after_probs}")

    except:
        accuracy_after = np.nan
        softmax_diff = np.nan

    print(f"Accuracy after SEU: {accuracy_after}")
    print("===================================")

    return {
        "accuracy_change": accuracy_after - initial_accuracy,
        "softmax_difference": softmax_diff,
        "absolute_difference": abs_diff
    }

In [58]:
run_seu(location_index=0,  # Change this to the index you want to test
        param_unique="AutoNormal",  # Change this to the parameter unique you want to test
        parameter_name="locs",  # Change this to "locs" or "scales"
        layer="conv1",  # Change this to the layer you want to test
        layer_module="weight",  # Change this to "weight" or "bias"
        bit_i=1,  # Change this to the bit index you want to test
        num_samples=num_samples
    )

Original value: -0.011526626534759998, SEU value: -3.922307759861827e+36, Abs difference: 3.922307759861827e+36


Evaluating: 100%|██████████| 50/50 [00:04<00:00, 10.80it/s]


Initial logits: [array([1037.9335, -883.6144], dtype=float32), array([ 329.1513, -479.1289], dtype=float32), array([ 263.64908, -140.95042], dtype=float32), array([ 1654.732 , -1220.5225], dtype=float32), array([ 996.47833, -671.7726 ], dtype=float32), array([ 633.1733 , -439.37286], dtype=float32), array([ 1359.1337, -2401.8782], dtype=float32), array([ 380.26993, -550.09314], dtype=float32), array([-87.18163,  61.53681], dtype=float32), array([ 656.74347, -431.4734 ], dtype=float32), array([-1247.1835,   729.6318], dtype=float32), array([ 267.85382, -256.91794], dtype=float32), array([-118.69012,  264.7318 ], dtype=float32), array([  790.07715, -1172.7738 ], dtype=float32), array([-430.4441 ,  550.94183], dtype=float32), array([ 437.4256 , -201.68456], dtype=float32), array([ 1738.2457, -1542.8749], dtype=float32), array([ 407.6357, -195.0756], dtype=float32), array([-286.551  ,  234.66348], dtype=float32), array([ 1101.4535 , -1013.86865], dtype=float32), array([ 2730.2856, -2470.40

{'accuracy_change': np.float64(-0.16125),
 'softmax_difference': np.float32(0.8907613),
 'absolute_difference': 3.922307759861827e+36}